In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/fingerprint_mlp_fold0.pth
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/test_predicted_fingerprints.npy
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/ATTRIBUTION.md
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_EXTRAS.csv
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_EXTRAS_METHOD.json
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_METHOD.json
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/BENCHMARK_SOURCES.csv
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/CHANGELOG.md
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/COVERAGE_SOURCES.json
/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior/FILE_DESCRIPTION

In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 104.9 MB/s  0:00:00


# 1. Configuración de rutas, hardware y librerías

In [3]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
import gc
from torch.utils.data import Dataset, DataLoader
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize


# --- CONFIGURACIÓN DE RUTAS ---
DATA_DIR = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
TRAIN_FOLDS_PATH = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.parquet')
TEST_PATH = os.path.join(DATA_DIR, 'test.parquet')

# Dataset donde guardaste los pesos del modelo del Notebook 2
MODEL_DIR = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/'
MODEL_PATH = os.path.join(MODEL_DIR, 'fingerprint_mlp_fold0.pth')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {DEVICE}")
print("Archivos disponibles en MODEL_DIR:", glob.glob(os.path.join(MODEL_DIR, '*')))

Usando dispositivo: cuda
Archivos disponibles en MODEL_DIR: ['/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/fingerprint_mlp_fold0.pth', '/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/test_predicted_fingerprints.npy']


# 2. Carga de datos

In [4]:
train_df = pd.read_parquet(TRAIN_PATH)
folds_df = pd.read_parquet(TRAIN_FOLDS_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Columnas de train_df:", list(train_df.columns))
print("Columnas de test_df:", list(test_df.columns))
print(f"Filas train: {len(train_df):,} | Filas folds: {len(folds_df):,} | Filas test: {len(test_df):,}")

Columnas de train_df: ['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'ms2_mzs', 'ms2_normalized_intensities', 'num_peaks', 'base_peak_intensity', 'collision_energy_ev', 'collision_energy_orig', 'collision_energy_orig_units']
Columnas de test_df: ['molecule_id', 'spectrum_id', 'ms2_mzs', 'ms2_normalized_intensities', 'base_peak_intensity', 'adduct', 'ionization_mode', 'instrument_type', 'precursor_mz', 'collision_energy_orig', 'collision_energy_ev', 'collision_energy_orig_units']
Filas train: 2,539,608 | Filas folds: 2,539,608 | Filas test: 1,213


# 3. Definición del modelo MLP y carga de fingerprint_mlp_fold0.pth

In [5]:
TOP_K = 64
OUTPUT_DIM = 1024
HIDDEN_DIM = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class TopKEncoderMLP(nn.Module):
    def __init__(self, top_k=64, input_dim=2, hidden_dim=512, output_dim=1024):
        super().__init__()
        self.top_k = top_k

        # Proyección inicial por cada pico (BatchNorm sobre hidden_dim, no sobre top_k)
        self.pico_mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # Capas globales para predecir el fingerprint de 1024 bits
        self.global_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, output_dim)  # Devuelve logits
        )

    def forward(self, x):
        # x tiene forma: (Batch_Size, Top_K, 2) [m/z, intensity]
        batch_size = x.size(0)

        # Aplanar para aplicar la capa lineal a todos los picos a la vez: (Batch_Size * Top_K, input_dim)
        x_flat = x.view(-1, x.size(-1))
        out_picos = self.pico_mlp(x_flat)

        # Reestructurar de vuelta a (Batch_Size, Top_K, hidden_dim)
        out_picos = out_picos.view(batch_size, self.top_k, -1)

        # Agregación global (Max Pooling a lo largo de los picos para capturar los fragmentos clave)
        global_repr, _ = torch.max(out_picos, dim=1)

        # Predicción final de los 1024 bits
        logits = self.global_mlp(global_repr)
        return logits


class TopKTestDataset(Dataset):
    def __init__(self, df, top_k=64):
        self.df = df.reset_index(drop=True)
        self.top_k = top_k

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mz = np.array(row['ms2_mzs'], dtype=np.float32)
        intensity = np.array(row['ms2_normalized_intensities'], dtype=np.float32)

        if len(mz) > 0:
            sorted_idx = np.argsort(intensity)[::-1]
            mz = mz[sorted_idx][:self.top_k]
            intensity = intensity[sorted_idx][:self.top_k]
            max_int = intensity.max()
            if max_int > 0:
                intensity = intensity / max_int

        current_len = len(mz)
        if current_len < self.top_k:
            mz = np.pad(mz, (0, self.top_k - current_len), 'constant', constant_values=0)
            intensity = np.pad(intensity, (0, self.top_k - current_len), 'constant', constant_values=0)

        spectrum_features = np.stack([mz, intensity], axis=-1)
        return torch.tensor(spectrum_features, dtype=torch.float32)


model = TopKEncoderMLP(top_k=TOP_K, input_dim=2, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM).to(device)
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print(f"✓ Modelo TopKEncoderMLP cargado desde: {MODEL_PATH}")

✓ Modelo TopKEncoderMLP cargado desde: /kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/fingerprint_mlp_fold0.pth


# 4. Predicción de fingerprints de 1024 bits en test.parquet

In [6]:
test_df = pd.read_parquet(TEST_PATH)
test_dataset = TopKTestDataset(test_df, top_k=TOP_K)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

test_preds = []
with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(device, non_blocking=True)
        if device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                logits = model(x_batch)
                probs = torch.sigmoid(logits)
        else:
            logits = model(x_batch)
            probs = torch.sigmoid(logits)
        test_preds.append(probs.detach().cpu().numpy().astype(np.float32))

test_pred_fps = np.concatenate(test_preds, axis=0)
np.save('class2_test_predicted_fingerprints.npy', test_pred_fps)
print(f"✓ Fingerprints predichos para test: {test_pred_fps.shape}")

✓ Fingerprints predichos para test: (1213, 1024)


# 5: Indexado de base química (PubChem / COCONUT) con GetMorganGenerator(radius=2, fpSize=1024)

In [7]:
RDLogger.DisableLog('rdApp.*')

for var_name in ['test_pred_fps', 'train_raw', 'folds_df', 'val_df', 'ref_fp_norm', 'val_probs']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()

PUBCHEM_DIR = '/kaggle/input/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior'
MASS_WINDOW_DA = 15.0
MAX_FILTERED_MOLS = 50_000
N_VAL_MOLS = 100

smiles_path = os.path.join(PUBCHEM_DIR, 'casmi26-pubchem-tier', 'pc_smiles.npy')
off_path = os.path.join(PUBCHEM_DIR, 'casmi26-pubchem-tier', 'pc_off.npy')
mass_path = os.path.join(PUBCHEM_DIR, 'casmi26-pubchem-tier', 'pc_mass.npy')
ik14_path = os.path.join(PUBCHEM_DIR, 'pc_ik14.npy')

if not os.path.exists(smiles_path):
    smiles_path = os.path.join(PUBCHEM_DIR, 'pc_smiles.npy')
    off_path = os.path.join(PUBCHEM_DIR, 'pc_off.npy')
    mass_path = os.path.join(PUBCHEM_DIR, 'pc_mass.npy')

# 1. Cargar muestra de 100 moléculas únicas de validación (Fold 0)
folds_df = pd.read_parquet(TRAIN_FOLDS_PATH, columns=['normalized_smiles', 'inchikey14', 'fold']).drop_duplicates()
val_keys_all = folds_df[folds_df['fold'] == 0]['inchikey14'].astype(str).str.split('-').str[0].dropna().unique()

rng = np.random.default_rng(42)
sampled_keys = set(rng.choice(val_keys_all, size=min(N_VAL_MOLS, len(val_keys_all)), replace=False))
del folds_df, val_keys_all
gc.collect()

val_df = pd.read_parquet(TRAIN_PATH)
val_df = val_df[val_df['inchikey14'].astype(str).str.split('-').str[0].isin(sampled_keys)].reset_index(drop=True)
val_df = val_df.assign(_ik14=val_df['inchikey14'].astype(str).str.split('-').str[0])
gc.collect()

val_target_keys = set(val_df['_ik14'].dropna())

# 2. Seleccionar candidatos en PubChem (hits de val_target_keys + candidatos por masa)
smiles_bytes = np.load(smiles_path, mmap_mode='r')
offsets = np.load(off_path, mmap_mode='r')
masses = np.load(mass_path, mmap_mode='r') if os.path.exists(mass_path) else None
pc_ik14 = np.load(ik14_path, mmap_mode='r') if os.path.exists(ik14_path) else None

n_total = len(offsets) - 1 if len(offsets) > 1 else len(offsets)

hit_indices = np.array([], dtype=np.int64)
if pc_ik14 is not None:
    n_k = min(n_total, len(pc_ik14))
    hit_list = []
    step = 1_000_000
    for start in range(0, n_k, step):
        end = min(start + step, n_k)
        chunk = pc_ik14[start:end]
        chunk_keys = np.array([
            (x.decode('utf-8', errors='ignore') if isinstance(x, bytes) else str(x)).split('-')[0]
            for x in chunk
        ])
        mask = np.isin(chunk_keys, list(val_target_keys))
        if np.any(mask):
            hit_list.append(np.flatnonzero(mask) + start)
    if hit_list:
        hit_indices = np.concatenate(hit_list)

if masses is not None and 'precursor_mz' in val_df.columns:
    mz_vals = pd.to_numeric(val_df['precursor_mz'], errors='coerce').dropna()
    q_low, q_high = np.percentile(mz_vals, [10, 90])
    min_mass = float(q_low) - MASS_WINDOW_DA
    max_mass = float(q_high) + MASS_WINDOW_DA
    n_m = min(n_total, len(masses))
    mass_indices = np.flatnonzero((masses[:n_m] >= min_mass) & (masses[:n_m] <= max_mass))
else:
    mass_indices = np.arange(n_total, dtype=np.int64)

remaining_slots = max(0, MAX_FILTERED_MOLS - len(hit_indices))
if remaining_slots > 0 and len(mass_indices) > 0:
    extra_indices = np.setdiff1d(mass_indices, hit_indices, assume_unique=False)[:remaining_slots]
    cand_indices = np.concatenate([hit_indices, extra_indices])
else:
    cand_indices = hit_indices[:MAX_FILTERED_MOLS]

print(f"Candidatos seleccionados: {len(cand_indices):,} (incluyendo {len(hit_indices):,} entradas matching de PubChem)")

# 3. Indexar fingerprints en float16
morgan_gen = GetMorganGenerator(radius=2, fpSize=OUTPUT_DIM)
ref_smiles = []
ref_fps = []

for idx_i in cand_indices:
    i = int(idx_i)
    start = int(offsets[i])
    end = int(offsets[i + 1]) if (i + 1) < len(offsets) else len(smiles_bytes)
    if end <= start:
        continue
    smi_str = bytes(smiles_bytes[start:end]).decode('utf-8', errors='ignore').strip('\x00 \r\n')
    if not smi_str:
        continue
    mol = Chem.MolFromSmiles(smi_str)
    if mol is None:
        continue
    fp = morgan_gen.GetFingerprintAsNumPy(mol).astype(np.float16)
    norm = np.linalg.norm(fp) + 1e-8
    ref_fps.append(fp / norm)
    ref_smiles.append(smi_str)

ref_fp_norm = np.vstack(ref_fps).astype(np.float16)
del ref_fps, smiles_bytes, offsets, masses, pc_ik14, cand_indices, hit_indices, mass_indices
gc.collect()

print(f"✓ Base indexada: {len(ref_smiles):,} moléculas | RAM matriz: {ref_fp_norm.nbytes / (1024**2):.1f} MB")

# 4. Predecir en validación, calcular similitud y agregar por _ik14
val_dataset = TopKTestDataset(val_df, top_k=TOP_K)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

model.eval()
val_probs_list = []
with torch.no_grad():
    for x_batch in val_loader:
        x_batch = x_batch.to(device)
        val_probs_list.append(torch.sigmoid(model(x_batch)).cpu().numpy().astype(np.float16))

val_probs = np.concatenate(val_probs_list, axis=0)
val_probs = val_probs / (np.linalg.norm(val_probs, axis=1, keepdims=True) + 1e-8)
del val_probs_list, val_dataset, val_loader
gc.collect()

TOP_K_PER_SPEC = 25
val_mol_scores = {}
ik14_list = val_df['_ik14'].tolist()

for i in range(len(val_probs)):
    sim = val_probs[i].astype(np.float32) @ ref_fp_norm.T.astype(np.float32)
    k_eff = min(TOP_K_PER_SPEC, len(sim))
    top_idx = np.argpartition(-sim, kth=k_eff - 1)[:k_eff]
    top_idx = top_idx[np.argsort(-sim[top_idx])]

    mol_key = ik14_list[i]
    d = val_mol_scores.setdefault(mol_key, {})
    for idx in top_idx:
        smi = ref_smiles[idx]
        d[smi] = d.get(smi, 0.0) + float(sim[idx])

del val_probs, ref_fp_norm, val_df
gc.collect()

y_pred_val = {
    mol_key: [smi for smi, _ in sorted(cand_map.items(), key=lambda kv: kv[1], reverse=True)[:25]]
    for mol_key, cand_map in val_mol_scores.items()
}
del val_mol_scores
gc.collect()

# 5. Calcular MRR@25 (conectividad vía InChIKey14 tras canonización tautomérica)
_ik_cache = {}
_taut_enum = rdMolStandardize.TautomerEnumerator()

def fast_smiles_to_ik14(smiles: str) -> str:
    if smiles in _ik_cache:
        return _ik_cache[smiles]
    if not isinstance(smiles, str) or not smiles.strip():
        _ik_cache[smiles] = ""
        return ""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            _ik_cache[smiles] = ""
            return ""
        canon_mol = _taut_enum.Canonicalize(mol)
        ik = Chem.MolToInchiKey(canon_mol)
        res = ik.split('-')[0] if ik else ""
    except Exception:
        res = ""
    _ik_cache[smiles] = res
    return res

ranks = []
for target_key, preds in y_pred_val.items():
    score = 0.0
    for rank_idx, pred_smiles in enumerate(preds[:25], start=1):
        if fast_smiles_to_ik14(pred_smiles) == target_key and target_key != "":
            score = 1.0 / rank_idx
            break
    ranks.append(score)

print(f"MRR@25 local en Fold 0 (100 moléculas, con candidatos matching incluidos): {np.mean(ranks):.5f}")

Candidatos seleccionados: 50,000 (incluyendo 263 entradas matching de PubChem)
✓ Base indexada: 50,000 moléculas | RAM matriz: 97.7 MB
MRR@25 local en Fold 0 (100 moléculas, con candidatos matching incluidos): 0.06143
